In [0]:
from pyspark.sql import SparkSession

# spark = SparkSession.builder.getOrCreate()

In [0]:
df=spark.read.option("header",True)\
    .option("inferSchema","True")\
    .option("mode","PERMISSIVE")\
    .csv("/Volumes/workspace/default/netflix/netflix_titles.csv")
df.display()

In [0]:
df.show(10)

In [0]:
#print coloumn names 
print(df.columns)

In [0]:
#count total rows
print("Total Rows:", df.count())


In [0]:
#display schema
df.printSchema()

In [0]:
#length of columns
print("Number of Columns:", len(df.columns))

In [0]:
#corrupted value detection
if "_corrupt_record" in df.columns:
    corrupt = df.filter(df["_corrupt_record"].isNotNull())

    if corrupt.count() > 0:
        print("Corrupted Records Found")
        corrupt.show(truncate=False)
    else:
        print("No corrupted records found.")
else:
    print("No _corrupt_record column present.")

In [0]:
#creating custome schema
from pyspark.sql.types import *

custom_schema = StructType([
    StructField("show_id", StringType(), True),
    StructField("type", StringType(), True),
    StructField("title", StringType(), True),
    StructField("director", StringType(), True),
    StructField("cast", StringType(), True),
    StructField("country", StringType(), True),
    StructField("date_added", StringType(), True),
    StructField("release_year", IntegerType(), True),
    StructField("rating", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("listed_in", StringType(), True),
    StructField("description", StringType(), True)
])

In [0]:
#read data again
df = spark.read \
    .option("header", True) \
    .schema(custom_schema) \
    .csv("/Volumes/workspace/default/netflix/netflix_titles.csv")
df.show(10)

In [0]:
#data transformation
#reenaming the coloumn
df = df.withColumnRenamed("listed_in", "genre")
df.show()

In [0]:
#filter rows
#keeep only movies
movies = df.filter(df.type == "Movie")
df.show()

In [0]:
#alias coloumns
movies.select(
    movies.title.alias("Movie_Title"),
    movies.release_year.alias("Year")
).show(10)

In [0]:
movies.count()

In [0]:
#add literal coloumns
from pyspark.sql.functions import lit

movies = movies.withColumn("Source", lit("Kaggle"))
df.show()

In [0]:
#add new calculated coloumns
from pyspark.sql.functions import when

movies = movies.withColumn(
    "Category",
    when(movies.release_year >= 2020, "New").otherwise("Old")
)
df.show()



In [0]:
#cast dataypes\
movies = movies.withColumn(
    "release_year",
    movies.release_year.cast("Integer")
)
df.show()


In [0]:
#drop unnessory coloumns
movies = movies.drop("description")
df.show()

In [0]:
#handling null values
movies = movies.fillna({
    "director": "Unknown",
    "country": "Not Available",
    "rating": "Not Rated"
})
df.show()

In [0]:
#removing duplicates
movies = movies.dropDuplicates()
df.show()

In [0]:
movies.count()

In [0]:
output_path = "/Volumes/workspace/default/netflix"

movies.write \
    .mode("overwrite") \
    .parquet(output_path)

In [0]:
final_df = spark.read.parquet(output_path)

final_df.show(5)